# Patient 360 Gold Build

Builds `caresync_dbws.gold.patient_360` from the STTM workbook `sttm_silver_to_gold.xlsx`. This notebook follows the documented grain of one row per patient and enriches patients with hospital, insurance, and lab-result aggregates.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS caresync_dbws.gold;

CREATE OR REPLACE TABLE caresync_dbws.gold.patient_360 AS
WITH lab_results_agg AS (
  SELECT
    patient_id,
    CAST(COUNT(*) AS INT) AS total_tests,
    CAST(SUM(CASE WHEN result_status = 'Abnormal' THEN 1 ELSE 0 END) AS INT) AS abnormal_test_count,
    CAST(SUM(CASE WHEN result_status = 'Critical' THEN 1 ELSE 0 END) AS INT) AS critical_test_count,
    MAX(test_date) AS last_test_date
  FROM caresync_dbws.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE) AS date_of_birth,
  CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name AS insurance_provider_name,
  COALESCE(l.total_tests, 0) AS total_tests,
  COALESCE(l.abnormal_test_count, 0) AS abnormal_test_count,
  COALESCE(l.critical_test_count, 0) AS critical_test_count,
  l.last_test_date
FROM caresync_dbws.silver.patients p
LEFT JOIN caresync_dbws.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN caresync_dbws.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_results_agg l
  ON p.patient_id = l.patient_id;


In [0]:
updated_df = spark.read.table("caresync_dbws.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))
